# Notebook A — Baseline question generation (template / heuristic)

This notebook builds a **non-neural baseline** for question generation: fixed **templates** over extracted **answer spans** (keywords) and resource **category**, matching the synthetic construction used in `build_learning_resources_qg.py`. There is **no Flan-T5 fine-tuning** here.

You will: load the same QG JSON as the T5 pipeline, apply a **reproducible train / validation / test split** (70% / 15% / 15%), generate template questions on the **validation** and **test** sets, compute **BLEU**, **ROUGE-L**, and **METEOR** with a bar chart for each split, and save **test** metrics to `baseline_metrics.json` for comparison with Notebook B (same test examples).

## 1. Environment: paths and imports

**What this section does:** Resolves the project `backend` folder (whether you run from `Educonnect` or `notebooks`), adds it to `sys.path`, and imports the same metric and plotting helpers used by `eval_quiz_t5.py` so results are comparable with the fine-tuned model.

In [4]:
import json
import random
import sys
from pathlib import Path


def find_backend():
    cwd = Path.cwd()
    for base in (cwd, cwd.parent, cwd.parent.parent):
        b = base / "backend"
        if (b / "eval_quiz_t5.py").exists():
            return b.resolve()
    raise FileNotFoundError(
        "Could not find backend/ — open this notebook from Educonnect or notebooks and run again."
    )


BACKEND = find_backend()
sys.path.insert(0, str(BACKEND))
DATA_DIR = BACKEND / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

from eval_quiz_t5 import compute_bleu, compute_rouge, compute_meteor, plot_metrics

print("BACKEND =", BACKEND)

BACKEND = C:\Users\USER\OneDrive\Desktop\The Actual Educonnect (2)\AI-Enabled-peer-to-peer-learning-framework\The Actual Educonnect\Educonnect\backend


## 2. Load the question-generation dataset

**What this section does:** Reads `learning_resources_qg.json` from `backend/data/`. If the file is missing, it builds pairs from learning resources (same as `build_learning_resources_qg.py`) so you can run end-to-end without a pre-exported file.

In [5]:
qg_path = DATA_DIR / "learning_resources_qg.json"

if not qg_path.exists():
    from build_learning_resources_qg import build_qg_pairs
    from load_learning_resources import get_learning_resources

    resources = get_learning_resources(enrich_quiz_source=True)
    pairs = build_qg_pairs(resources, max_per_resource=5)
    raw = [
        {"context": c, "answer": a, "question": q, "category": cat}
        for c, a, q, cat in pairs
    ]
    qg_path.write_text(json.dumps(raw, indent=2), encoding="utf-8")
    print("Wrote", qg_path)
else:
    raw = json.loads(qg_path.read_text(encoding="utf-8"))

print("Loaded rows:", len(raw))

Loaded rows: 355


## 3. Filter rows and create train / validation / test split

**What this section does:** Keeps only rows where the **answer substring appears in the context** (required for highlight-style QG). Then shuffles index order with a **fixed seed**, assigns **70% / 15% / 15%** to train, validation, and test, and writes `qg_split_manifest.json` so Notebook B can reuse the **exact same test set** for a fair comparison.

In [6]:
valid_data = []
for item in raw:
    ctx, ans = item["context"], item["answer"]
    if ans and ans in ctx:
        valid_data.append(item)

SPLIT_SEED = 42
rng = random.Random(SPLIT_SEED)
n = len(valid_data)
indices = list(range(n))
rng.shuffle(indices)

n_train = int(0.70 * n)
n_val = int(0.15 * n)
train_ix = indices[:n_train]
val_ix = indices[n_train : n_train + n_val]
test_ix = indices[n_train + n_val :]

manifest = {
    "seed": SPLIT_SEED,
    "n_valid": n,
    "train": train_ix,
    "val": val_ix,
    "test": test_ix,
}
manifest_path = DATA_DIR / "qg_split_manifest.json"
manifest_path.write_text(json.dumps(manifest), encoding="utf-8")
print(f"Train={len(train_ix)}  Val={len(val_ix)}  Test={len(test_ix)}  -> {manifest_path}")

val_items = [valid_data[i] for i in val_ix]
test_items = [valid_data[i] for i in test_ix]

Train=220  Val=47  Test=48  -> C:\Users\USER\OneDrive\Desktop\The Actual Educonnect (2)\AI-Enabled-peer-to-peer-learning-framework\The Actual Educonnect\Educonnect\backend\data\qg_split_manifest.json


## 4. Baseline: template-based question generator (no neural model)

**What this section does:** Defines the same **template strings** as `build_learning_resources_qg.py` and a small function that fills them using the **answer span** and **category**. Template choice is **deterministic** per row index (`index % len(templates)`) so runs are reproducible. The next section applies this to validation and test rows.

In [7]:
TEMPLATES = [
    "What does the term {} refer to when studying {}?",
    "Which concept is best captured by {} when studying {}?",
    "According to this material, what does {} refer to within {}?",
    "How does {} connect to the subject of {}?",
    "What should a learner take away about {} in {}?",
]


def baseline_question(answer: str, category: str, row_index: int) -> str:
    cat = (category or "general").replace("-", " ").title()
    tpl = TEMPLATES[row_index % len(TEMPLATES)]
    return tpl.format(answer, cat)

## 5. Evaluation on validation and test (metrics + plots)

**What this section does:** Runs the baseline on the **validation** split first (metrics + `baseline_val_eval_results.png`, `baseline_val_metrics.json`), then on the **test** split. Test metrics are saved as **`baseline_metrics.json`** — Notebook B uses the **same test indices** so you can compare Flan-T5 vs baseline fairly. Metrics are **BLEU**, **ROUGE-L**, and **METEOR** (standard for text generation; there is no confusion matrix because outputs are open-ended strings, not discrete class labels).

In [9]:
def baseline_preds_refs(items):
    preds, refs = [], []
    for i, item in enumerate(items):
        cat = item.get("category", "general")
        preds.append(baseline_question(item["answer"], cat, i))
        refs.append(item["question"])
    return preds, refs


def score_split(name, preds, refs):
    m = {
        "BLEU": compute_bleu(preds, refs),
        "ROUGE-L": compute_rouge(preds, refs)["rougeL_fmeasure"],
        "METEOR": compute_meteor(preds, refs),
    }
    print(f"\n--- {name} ---")
    for k, v in m.items():
        print(f"{k:12} {v:.4f}")
    return m


# Validation
vp, vr = baseline_preds_refs(val_items)
val_metrics = score_split("Validation", vp, vr)
(DATA_DIR / "baseline_val_metrics.json").write_text(
    json.dumps(val_metrics, indent=2), encoding="utf-8"
)
plot_metrics(
    val_metrics,
    DATA_DIR / "baseline_val_eval_results.png",
    title="Baseline QG (template) — validation set",
)

# Test (used for comparison with Notebook B)
tp, tr = baseline_preds_refs(test_items)
metrics = score_split("Test", tp, tr)
(DATA_DIR / "baseline_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
plot_metrics(
    metrics,
    DATA_DIR / "baseline_eval_results.png",
    title="Baseline QG (template) — test set",
)
print("\nSaved baseline_metrics.json (test) for Notebook B comparison.")


--- Validation ---
BLEU         0.2616
ROUGE-L      0.0000
METEOR       0.3862
Saved chart to C:\Users\USER\OneDrive\Desktop\The Actual Educonnect (2)\AI-Enabled-peer-to-peer-learning-framework\The Actual Educonnect\Educonnect\backend\data\baseline_val_eval_results.png

--- Test ---
BLEU         0.3170
ROUGE-L      0.0000
METEOR       0.4121
Saved chart to C:\Users\USER\OneDrive\Desktop\The Actual Educonnect (2)\AI-Enabled-peer-to-peer-learning-framework\The Actual Educonnect\Educonnect\backend\data\baseline_eval_results.png

Saved baseline_metrics.json (test) for Notebook B comparison.
